In [76]:
# Imports
import astropy.units as u
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import scipy
from datetime import date
import time

matplotlib.use("tkagg")

In [2]:
# Define constants
sag_a_r_s = 1.2 * 10 ** 10 * u.m  # https://academic.oup.com/mnras/article/421/2/1315/1134797?login=false
r_g = sag_a_r_s / 2
c = 2.99 * 10 ** 8 * (u.m / u.s)

In [3]:
def q_squared_to_b_squared(q, v, r_s):
    """Convert from periastron to impact parameter b"""
    gamma = 1 / np.sqrt(1 - (v / c) ** 2)
    b_squared = q ** 2 * (c / v) ** 2 * (1 / (1 - r_s / q) - 1 / gamma ** 2)
    return b_squared

In [71]:
def process_time_step_orbit(r, dt, energy_term, h):
    """Calculate changes to r and the vaporization loss"""
    dr = -1 * np.sqrt(energy_term - (1 - sag_a_r_s / r) * (c ** 2 + h ** 2 / r ** 2)) * dt
    return dr

last_time = 0

def process_time_step_orbit_solve_ivp(t, r_from_q, energy_term, h, q):
    """Calculate changes to r and the vaporization loss, modified for solve_ivp"""
    # global last_time
    # print(t - last_time)
    # last_time = t
    r = abs(r_from_q * u.m) + q
    dr = -1 * np.sqrt(energy_term - (1 - sag_a_r_s / r) * (c ** 2 + h ** 2 / r ** 2))
    return dr.value + 0.01


def calc_radius_loss(distance, dt):
    dr_frag = 10 * u.m / (200 * u.s) * (distance / (10 * r_g)) ** -3 * dt
    return dr_frag


def check_if_correct_q(q, b, v):
    q = q * r_g
    total = (b ** 2 - q ** 2 * c ** 2 / v ** 2 * (1 / (1 - sag_a_r_s / q) - (1 - (v / c) ** 2))).value
    return total ** 2


def numerically_determine_q(b, v):
    """Numerically determine the closest approach"""
    """Has issues"""
    result = scipy.optimize.minimize(check_if_correct_q, np.array([8]), args=(b, v), bounds=[(4.01, None)], method="Nelder-Mead")
    return result.x

In [5]:
def integrate_orbit(q, v_0, dt=0.1 * u.s, output=False):
    """Integrate the radius at a given time for an orbit with the given parameters."""
    steps_done = 0
    b_0 = np.sqrt(q_squared_to_b_squared(q, v_0, sag_a_r_s))
    d_0 = 30 * r_g
    orbit_energy = c ** 2 * 1 / (1 - (v_0 / c) ** 2)
    h = v_0 * b_0 * 1 / np.sqrt(1 - (v_0 / c) ** 2)
    distances = np.array([d_0.value]) * u.m
    # q = numerically_determine_q(b_0, v_0)[0] * r_g
    # if np.log10(check_if_correct_q(q / r_g, b_0, v_0)) > 50:
    #     print(f"Orbit: {b_0}, {v_0} has undefined q")
    #     return 0, 0, False
    # if q >= 30 * r_g:
    #     print(f"Orbit: {b_0}, {v_0} doesn't tidally disrupt")
    #     return 0, 0, False
    # print(check_if_correct_q(q / r_g, b_0, v_0), q / r_g)

    # Integrate the orbit to find distance at each t
    print(d_0, q)
    while d_0 > q:
        delta_r = process_time_step_orbit(d_0, dt, orbit_energy, h)
        d_0 += delta_r
        distances = np.append(distances, d_0)
        steps_done += 1
        if steps_done % 100 == 0 and output:
            print(steps_done, d_0 / r_g)

    times = np.arange(steps_done + 1) * dt
    times_since_periastron = times - times[-1]

    return distances, times_since_periastron, True

In [6]:
def simulate_store_orbit(q_0, v_0, output=False):
    """Simulate and record the integrated orbit"""
    distances, times_since_periastron, success = integrate_orbit(q_0, v_0, output=output)
    print(len(times_since_periastron))
    if not success:
        return
    orbits_completed_file = "Text_files/orbits_integrated.csv"
    integrations_file = "Text_files/integrations.csv"

    with open(orbits_completed_file, "a+") as orbits_completed:
        orbits_completed.seek(0)
        existing_orbits = orbits_completed.readlines()
        if f"{date.today()},{q_0},{v_0}\n" in existing_orbits:
            print("orbit exists")
            return

        with open(integrations_file, "a") as integrations:
            rows = [f"{date.today()},{q_0.value},{v_0.value},{t:.1f},{d:.0f}" for t, d in np.column_stack((times_since_periastron.value, distances.value))]
            text_to_write = "\n".join(rows)
            integrations.write(text_to_write + "\n")
            integrations.close()

        orbits_completed.write(f"{date.today()},{q_0},{v_0}\n")
        orbits_completed.close()


In [36]:
# Generate and record many orbits to build a database
vs_to_test = np.logspace(0, 6, 10) * u.m / u.s
qs_to_test = np.linspace(4.1, 30, 10) * r_g

for current_v in vs_to_test:
    for current_q in qs_to_test:
        print(current_q / r_g, current_v)
        simulate_store_orbit(current_q, current_v, output=False)

4.1 1.0 m / s
180000000000.0 m 24599999999.999996 m
24443
6.977777777777777 1.0 m / s
180000000000.0 m 41866666666.666664 m
22305
9.855555555555554 1.0 m / s
180000000000.0 m 59133333333.33333 m
22945
12.733333333333333 1.0 m / s
180000000000.0 m 76400000000.0 m
23363
15.611111111111112 1.0 m / s
180000000000.0 m 93666666666.66667 m
23320
18.488888888888887 1.0 m / s
180000000000.0 m 110933333333.33333 m
22663
21.366666666666667 1.0 m / s
180000000000.0 m 128200000000.0 m
21201
24.244444444444447 1.0 m / s
180000000000.0 m 145466666666.6667 m
18601
27.12222222222222 1.0 m / s
180000000000.0 m 162733333333.3333 m
14067
30.0 1.0 m / s
180000000000.0 m 180000000000.0 m
1
4.1 4.641588833612778 m / s
180000000000.0 m 24599999999.999996 m
24443
6.977777777777777 4.641588833612778 m / s
180000000000.0 m 41866666666.666664 m
22305
9.855555555555554 4.641588833612778 m / s
180000000000.0 m 59133333333.33333 m
22945
12.733333333333333 4.641588833612778 m / s
180000000000.0 m 76400000000.0 m
2336

In [38]:
# Investigate how timestep matters
q, v = 5 * r_g, 1770 * u.m / u.s
dts = [1, 0.1, 0.001]
plt.figure()
for time_step in dts:
    distances, times_since_periastron, success = integrate_orbit(q, v, time_step * u.s)
    plt.plot(times_since_periastron - times_since_periastron[0], distances, label=f"{time_step}s")

plt.show()

180000000000.0 m 30000000000.0 m
180000000000.0 m 30000000000.0 m
180000000000.0 m 30000000000.0 m


In [7]:
# Read in orbits from text file
def get_specific_orbit(q_to_get, v_to_get, orbit_file):
    """Retrieve the distances and times for a given orbit, using date, q and v"""
    all_orbits = np.loadtxt(orbit_file, delimiter=",", usecols=(1, 2, 3, 4))
    correct_orbit = all_orbits[all_orbits[:, 0] == q_to_get][all_orbits[all_orbits[:, 0] == q_to_get][:, 1] == v_to_get]
    print(f"The given orbit has {len(correct_orbit[:, 0])} matching points")
    return correct_orbit[:, 2], correct_orbit[:, 3]


# Read in all orbits from text file and format as a list of orbits
def get_all_orbits(orbit_file):
    """Get all orbits from orbit file"""
    all_orbits = np.loadtxt(orbit_file, delimiter=",", usecols=(1, 2, 3, 4))
    processed_orbits = []
    current_orbit = all_orbits[0, 0:2]
    current_orbit_data = np.array([0, 0, 0, 0])
    for row_index in range(len(all_orbits[:, 0])):
        if not np.array_equal(all_orbits[row_index, 0:2], current_orbit):
            processed_orbits.append(current_orbit_data[1:, :])
            current_orbit_data = np.array([0, 0, 0, 0])
            current_orbit = all_orbits[row_index, 0:2]

        current_orbit_data = np.vstack((current_orbit_data, all_orbits[row_index, :]))

    return processed_orbits

In [9]:
# Test getting in all orbits
integrated_orbits = get_all_orbits("./Text_files/integrations.csv")
for orbit in integrated_orbits:
    print(orbit[0, 1], orbit[0, 2])

1.0 -2444.2
1.0 -2230.4
1.0 -2294.4
1.0 -2336.2
1.0 -2331.9
1.0 -2266.2
1.0 -2120.0
1.0 -1860.0
1.0 -1406.6
1.0 0.0
4.641588833612778 -2444.2
4.641588833612778 -2230.4
4.641588833612778 -2294.4
4.641588833612778 -2336.2
4.641588833612778 -2331.9
4.641588833612778 -2266.2
4.641588833612778 -2120.0
4.641588833612778 -1860.0
4.641588833612778 -1406.6
4.641588833612778 0.0
21.544346900318832 -2444.2
21.544346900318832 -2230.4
21.544346900318832 -2294.4
21.544346900318832 -2336.2
21.544346900318832 -2331.9
21.544346900318832 -2266.2
21.544346900318832 -2120.0
21.544346900318832 -1860.0
21.544346900318832 -1406.6
21.544346900318832 0.0
100.0 -2444.2
100.0 -2230.4
100.0 -2294.4
100.0 -2336.2
100.0 -2331.9
100.0 -2266.2
100.0 -2120.0
100.0 -1860.0
100.0 -1406.6
100.0 0.0
464.15888336127773 -2444.2
464.15888336127773 -2230.4
464.15888336127773 -2294.4
464.15888336127773 -2336.2
464.15888336127773 -2331.9
464.15888336127773 -2266.2
464.15888336127773 -2120.0
464.15888336127773 -1860.0
464.158883

In [81]:
# Integrate over an orbit using solve_ivp and return its luminosity curve
def integrate_luminosity(times, v_inf, q, progenitor_radius, vaporization_fudge_factor, boulder_size_power_law, perihelion_time, rtol, atol):

    # Other parameters
    distance_from_q = 30 * r_g - q
    initial_radius = np.linspace(10, 150, 1000) * u.m
    rho = 100 * u.kg / (u.m ** 3)
    epsilon = 0.1
    b_inf = np.sqrt(q_squared_to_b_squared(q, v_inf, sag_a_r_s))
    energy = c ** 2 * 1 / (1 - (v_inf / c) ** 2)
    angular_momentum_h = v_inf * b_inf * 1 / np.sqrt(1 - (v_inf / c) ** 2)

    # Integrate the orbit to find distance at each t
    # while num_steps < 4000:
    #     change_r = process_time_step_orbit(abs(distance_from_q) + q, time_step, energy, angular_momentum_h)
    #     distance_from_q += change_r
    #     radiuses = np.append(radiuses, abs(distance_from_q) + q)
    #     num_steps += 1
    #     if num_steps % 100 == 0:
    #         print(num_steps, (abs(distance_from_q) + q) / r_g)

    print("int started")
    start = time.time()
    ode_result = scipy.integrate.solve_ivp(process_time_step_orbit_solve_ivp, (0, 4466), [distance_from_q.value], t_eval=times, args=(energy, angular_momentum_h, q), rtol=rtol, atol=atol)
    end = time.time()
    print(end - start)
    print("int finished")

    radius_loss_at_each_t = calc_radius_loss(ode_result.y[0, :] * u.m, 1 * u.s)
    total_radius_lost = np.cumulative_sum(radius_loss_at_each_t)
    radiuses_at_each_time = np.transpose(np.tile(total_radius_lost, (len(initial_radius), 1)))
    radius_at_t = np.add(-1 * radiuses_at_each_time, initial_radius)
    radius_at_t[radius_at_t < 0] = 0
    luminosities = (epsilon * c.value ** 2) * 4 * np.pi * rho * radius_at_t ** 2 * np.transpose(np.tile(radius_loss_at_each_t, (len(initial_radius), 1)))
    scale_factors = initial_radius ** -boulder_size_power_law
    total_volume = (4/3) * np.pi * progenitor_radius ** 3
    unscaled_fragment_volume = np.sum(scale_factors * (4/3) * np.pi * initial_radius ** 3)
    scale_factors_volume_adjusted = scale_factors * total_volume / unscaled_fragment_volume
    scaled_luminosities = luminosities * scale_factors_volume_adjusted[None, :]
    return np.sum(scaled_luminosities, axis=1)

In [82]:
# Test the integrate luminosity function
brightness_6 = integrate_luminosity(np.linspace(0, 4000, 4000), 180 * u.m / u.s, 5 * r_g, 5000 * u.m, 1, 1.8, 0, 10 **-10, 10**-5)
brightness_8 = integrate_luminosity(np.linspace(0, 4000, 4000), 180 * u.m / u.s, 5 * r_g, 5000 * u.m, 1, 1.8, 0, 10 **-6, 10**-4)
brightness_4 = integrate_luminosity(np.linspace(0, 4000, 4000), 180 * u.m / u.s, 5 * r_g, 5000 * u.m, 1, 1.8, 0, 10 **-6, 10**-2)
plt.figure()
plt.plot(brightness_4, label="absolute tolerance=10^-2")
plt.plot(brightness_6, label="absolute tolerance=10^-6")
plt.plot(brightness_8, label="absolute tolerance=10^-4")
plt.legend()
plt.xlabel("Time")
plt.ylabel("Luminosity")
plt.show()

int started
20.56751585006714
int finished
int started
6.164234399795532
int finished
int started
5.530729532241821
int finished


In [67]:
# Experiment with different variables
qs_to_test = np.linspace(5, 29, 5)
plt.figure()
for q in qs_to_test:
    plt.plot(integrate_luminosity(np.linspace(0, 4000, 4000), 180 * u.m / u.s, q * r_g, 5000 * u.m, 1, 1.8, 0), label=f"q={q}")
    print(q)
plt.legend()
plt.xlabel("Time")
plt.ylabel("Luminosity")
plt.show()

int started
0.0
22.575250839438734
-22.56588047434238
0.004685182548177614
0.023425912740888087
0.00416460670949121
0.005205758386864018
0.0
0.09370365096355235
0.04685182548177613
0.23425912740888089
0.04164606709491203
0.052057583868640234
0.0
0.9370365096355233
0.46851825481776155
2.342591274088808
0.416460670949121
0.5205758386864021
0.0
9.370365096355233
4.685182548177615
23.425912740888084
4.16460670949121
5.205758386864019
0.0
93.7036509635523
46.85182548177619
234.25912740888077
41.6460670949121
52.05758386864022
0.0
-392.5003163597868
38.00896922898738
190.04484614493694
33.78575042576654
42.23218803220823
0.0
76.01793845797476
38.008969228987326
190.04484614493697
33.78575042576654
42.23218803220823
0.0
-324.3793468810142
27.855172704429776
139.27586352214905
24.760153515048614
30.95019189381094
0.0
55.71034540885955
27.855172704429833
139.27586352214905
24.760153515048614
30.95019189381094
0.0
-234.4977503103845
22.026988366956857
110.13494183478383
19.579545215072585
24.474

KeyboardInterrupt: 